# Mutation Walkthrough

This notebook walks through the current symbolic-integration tree pipeline in `tree_diffusion_integration`:

`prefix expression -> AST -> canonical form -> position index -> mutation operators -> current observation state -> reverse edit target`

The most important functions in that pipeline are `parse_prefix_string(...)`, `serialize_prefix_string(...)`, `canonicalize(...)`, `index_tree_positions(...)`, `local_replacement_candidates(...)`, `can_locally_replace(...)`, `local_replace_once(...)`, `sample_valid_subtree(...)`, `can_sampled_subtree_replace(...)`, `replace_subtree_by_node_id(...)`, `collect_candidate_nodes(...)`, `mutate_once(...)`, `build_observation(...)`, `first_edit_toward_target(...)`, and `compute_edit_path(...)`.

The goal is to make each stage concrete: what each function takes as input, what it returns, what invariants it enforces, and why the later mutation stages depend on the earlier normalization steps.

The notebook is written as a runnable tutorial and intentionally stores no executed outputs.


In [ ]:
from pathlib import Path
import sys
import random
from pprint import pprint
from dataclasses import asdict


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src").is_dir() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(
        "Could not find the tree_diffusion_integration repo root from the current working directory."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Using repo root: {REPO_ROOT}")

from src.mathlang.ast import BinaryOp, Const, UnaryOp, Var
from src.mathlang.parser import parse_prefix_string
from src.mathlang.serializer import serialize_prefix_string
from src.mathlang.canonicalize import canonicalize
from src.tree_diffusion.positions import index_tree_positions
from src.tree_diffusion.mutation_grammar import (
    can_locally_replace,
    can_sampled_subtree_replace,
    local_replacement_candidates,
)
from src.tree_diffusion.mutation import (
    collect_candidate_nodes,
    local_replace_once,
    mutate_once,
    replace_subtree_by_node_id,
    sample_valid_subtree,
)
from src.tree_diffusion.observation import DEFAULT_PROBE_POINTS, build_observation


## Example Expressions And ASTs

We start with a few concrete prefix expressions, parse them into the math AST classes, and print both the raw dataclass representation and a small recursive tree view.

`parse_prefix_string(...)` is the entry point from notebook-friendly text into the typed AST used everywhere else in the mutation code. It tokenizes on whitespace, recognizes `INT+` / `INT-` followed by digit tokens as numeric constants, parses unary operators into `UnaryOp`, and parses every binary operator, including `add` and `mul`, into `BinaryOp`.

`serialize_prefix_string(...)` is the inverse view we use throughout the walkthrough. It turns the AST back into the prefix token stream that the notebook prints in before / after comparisons, so it is the easiest way to see how a mutation changed structure. Numeric constants round-trip through the serializer's token format, including fractions, and a parsed constant `div` can already collapse to a `Const` when both sides are numeric and the denominator is nonzero.


In [ ]:
EXAMPLE_EXPRESSIONS = [
    "sin x",
    "pow x INT+ 2",
    "mul x mul INT+ 1 INT+ 2",
    "add sin x pow x INT+ 2",
]


def expr_tree_lines(node, indent: str = "") -> list[str]:
    if isinstance(node, Const):
        if node.is_named:
            return [f"{indent}Const(symbol={node.symbol!r})"]
        return [f"{indent}Const(value={node.value})"]

    if isinstance(node, Var):
        return [f"{indent}Var(name={node.name!r})"]

    if isinstance(node, UnaryOp):
        lines = [f"{indent}UnaryOp(op={node.op!r})", f"{indent}  operand:"]
        lines.extend(expr_tree_lines(node.operand, indent + "    "))
        return lines

    if isinstance(node, BinaryOp):
        lines = [f"{indent}BinaryOp(op={node.op!r})", f"{indent}  left:"]
        lines.extend(expr_tree_lines(node.left, indent + "    "))
        lines.append(f"{indent}  right:")
        lines.extend(expr_tree_lines(node.right, indent + "    "))
        return lines

    raise TypeError(f"Unsupported node type: {type(node)!r}")


for expression in EXAMPLE_EXPRESSIONS:
    parsed = parse_prefix_string(expression)
    print("=" * 80)
    print(f"Input prefix:            {expression}")
    print(f"Parsed dataclass repr:   {parsed!r}")
    print(f"Serialized prefix:       {serialize_prefix_string(parsed)}")
    print("Tree view:")
    print("\n".join(expr_tree_lines(parsed)))


## Canonicalization And Position Indexing

`canonicalize(...)` is the first normalization pass applied before mutation. It does not try to prove arbitrary algebraic equivalence; it rewrites the AST into one consistent structural form that the mutation code can index and compare reliably.

Concretely, `canonicalize(...)`:

- normalizes operator and named-constant tokens,
- recursively canonicalizes children first,
- flattens same-operator binary `add` / `mul` chains into a temporary term list,
- sorts commutative associative terms by a structural key and rebuilds a right-nested `BinaryOp` chain,
- folds numeric `div(const, const)` into a single `Const` when the denominator is nonzero,
- strips top-level additive constants that do not contain `x`, so `add INT+ 7 add x INT+ 2` becomes just `x`.

Mutation always starts from the canonical tree, not the raw parsed tree. That matters because node ids, token spans, subtree sizes, and equality checks are all defined on this normalized form.

`index_tree_positions(...)` then walks the canonical tree in preorder and records one `NodePosition` per node. Each row tells us:

- `node_id`: the preorder id later used by subtree replacement,
- `token_start` / `token_end`: the half-open span in the serialized canonical prefix token stream,
- `depth`: root depth is 0,
- `parent_id`: the preorder id of the parent node,
- `path`: a tuple of child labels from the root to the node.


In [ ]:
def print_table(rows: list[dict], columns: list[str]) -> None:
    if not rows:
        print("<no rows>")
        return

    widths = {
        column: max(len(column), max(len(str(row[column])) for row in rows))
        for column in columns
    }
    header = " | ".join(f"{column:<{widths[column]}}" for column in columns)
    divider = "-+-".join("-" * widths[column] for column in columns)
    print(header)
    print(divider)
    for row in rows:
        print(" | ".join(f"{str(row[column]):<{widths[column]}}" for column in columns))


def show_canonicalization(expression: str) -> None:
    original = parse_prefix_string(expression)
    canonical = canonicalize(original)
    print("-" * 80)
    print(f"Original:  {expression}")
    print(f"Canonical: {serialize_prefix_string(canonical)}")


for expression in EXAMPLE_EXPRESSIONS:
    show_canonicalization(expression)

print("-" * 80)
extra = "add INT+ 7 add x INT+ 2"
print(f"Extra simplification example: {extra}")
print(f"Canonical: {serialize_prefix_string(canonicalize(parse_prefix_string(extra)))}")

print("\nPosition index for 'add sin x pow x INT+ 2' with sigma_small=2:")
index_expr = canonicalize(parse_prefix_string("add sin x pow x INT+ 2"))
index = index_tree_positions(index_expr, sigma_small=2)
print(f"Serialized tokens: {tuple(serialize_prefix_string(index_expr).split())}")

rows = []
for position in index.positions:
    row = asdict(position)
    row["subtree"] = serialize_prefix_string(index.node_id_to_node[position.node_id])
    rows.append(row)

print_table(
    rows,
    [
        "node_id",
        "parent_id",
        "depth",
        "production_family",
        "op",
        "token_start",
        "token_end",
        "subtree_size",
        "is_mutable",
        "subtree",
    ],
)


## Local Same-Shape Replacement

Local replacement is the conservative mutation path. It changes the selected node in place without resampling its whole interior.

`local_replacement_candidates(node)` returns abstract replacement specs, not concrete AST nodes. For leaves, the spec says which leaf kind is allowed (`numeric_const`, `named_const`, or `var`). For operators, the spec fixes the replacement shape, operator label, and child count. A later step materializes one of those specs into an actual replacement expression.

`can_locally_replace(source, candidate)` enforces the exact local invariants:

- `Leaf <-> Leaf`
- `UnaryOp <-> UnaryOp`
- `BinaryOp <-> BinaryOp`

For operator nodes, the children must stay exactly the same and only the root label changes. For leaf nodes, the replacement must still be a leaf, must differ from the original node, and `Var` is limited to `x`. Since `add` and `mul` are ordinary binary operators now, they can participate in binary local replacements when the two children are unchanged.

`local_replace_once(expr, selected_node_id, rng)` canonicalizes the input, reindexes it, looks up the selected current-tree node, chooses a legal local replacement candidate, materializes it, replaces the subtree by node id, and canonicalizes the final result.


In [ ]:
def format_spec(spec) -> str:
    if spec.leaf_kind is not None:
        return f"leaf:{spec.leaf_kind}"
    return f"{spec.shape}:{spec.op} (child_count={spec.child_count})"


def show_local_candidates(label: str, node) -> None:
    print("-" * 80)
    print(label)
    print(f"Node: {serialize_prefix_string(node)}")
    print([format_spec(spec) for spec in local_replacement_candidates(node)])


const_leaf = parse_prefix_string("INT+ 2")
var_leaf = parse_prefix_string("x")
unary_expr = parse_prefix_string("sin x")
pow_expr = parse_prefix_string("pow x INT+ 2")
add_expr = parse_prefix_string("add x pow x INT+ 2")

show_local_candidates("Leaf constant candidates", const_leaf)
show_local_candidates("Variable x candidates", var_leaf)
show_local_candidates("Unary candidates", unary_expr)
show_local_candidates("Binary pow candidates", pow_expr)
show_local_candidates("Binary add candidates", add_expr)

print()
print("Selected can_locally_replace(...) checks:")
checks = [
    (
        "legal unary: sin(x) -> cos(x)",
        parse_prefix_string("sin x"),
        parse_prefix_string("cos x"),
    ),
    (
        "legal binary: pow(x, 2) -> div(x, 2)",
        parse_prefix_string("pow x INT+ 2"),
        parse_prefix_string("div x INT+ 2"),
    ),
    (
        "legal binary add->mul with same children",
        parse_prefix_string("add x pow x INT+ 2"),
        parse_prefix_string("mul x pow x INT+ 2"),
    ),
    (
        "illegal binary replacement with changed child",
        parse_prefix_string("add x pow x INT+ 2"),
        parse_prefix_string("mul x INT+ 2"),
    ),
]

for label, source, target in checks:
    print("-" * 80)
    print(label)
    print(f"source: {serialize_prefix_string(source)}")
    print(f"target: {serialize_prefix_string(target)}")
    print(f"can_locally_replace: {can_locally_replace(source, target)}")


In [ ]:
def show_local_mutation(label: str, expr, selected_node_id: int, seed: int) -> None:
    result = local_replace_once(expr, selected_node_id=selected_node_id, rng=random.Random(seed))
    if result is None:
        raise RuntimeError(f"local_replace_once returned None for {label}")

    print("-" * 80)
    print(label)
    print(f"Canonical input expression: {serialize_prefix_string(canonicalize(expr))}")
    print(f"Selected node id:          {selected_node_id}")
    print(f"Original subtree:          {serialize_prefix_string(result.original_subtree)}")
    print(f"Replacement subtree:       {serialize_prefix_string(result.replacement_subtree)}")
    print(f"Final mutated expression:  {serialize_prefix_string(result.mutated_expr)}")


show_local_mutation(
    "Leaf replacement (constant leaf)",
    parse_prefix_string("pow x INT+ 5"),
    selected_node_id=2,
    seed=1,
)
show_local_mutation(
    "Unary replacement",
    parse_prefix_string("sin x"),
    selected_node_id=0,
    seed=1,
)
show_local_mutation(
    "Binary pow replacement",
    parse_prefix_string("pow x INT+ 2"),
    selected_node_id=0,
    seed=0,
)
show_local_mutation(
    "Binary add/mul replacement",
    parse_prefix_string("add x pow x INT+ 2"),
    selected_node_id=0,
    seed=0,
)


## Sampled Small Subtree Replacement

Sampled subtree replacement is the broader structural path. Instead of only changing the root label of the selected node, it can resample an entirely new subtree for the same production family and then splice that subtree into the canonical expression.

`sample_valid_subtree(family, sigma_small, rng)` is the constructor for these proposals. The `family` argument chooses the outer kind of subtree (`CONST`, `UNARY_EXPR`, `ADD_EXPR`, `MUL_EXPR`, `POW_EXPR`, `DIV_EXPR`, or the more general `EXPR` helper used internally), and `sigma_small` acts like a size budget for how much non-leaf structure may be generated beneath that root. Some families have extra rules: sampled `pow` subtrees often keep a constant exponent, and sampled `div` subtrees coerce away zero denominators.

`can_sampled_subtree_replace(source, candidate)` is looser than `can_locally_replace(...)`, but it still enforces production-family compatibility. A `pow` node can be replaced by another `pow` subtree with richer descendants, an `add` node by another `add`, and a numeric constant only by another numeric constant. Variables currently do not take the sampled subtree path.

`replace_subtree_by_node_id(expr, node_id, replacement)` does the actual tree surgery. It walks the tree using the same preorder numbering scheme as `index_tree_positions(...)` and swaps in the replacement when it reaches the requested node id. By itself, this function does not check whether the replacement is legal and it does not canonicalize the result; the caller is responsible for both of those steps.


In [ ]:
original = parse_prefix_string("pow x sin x")
candidate = parse_prefix_string("pow x add x INT+ 1")
sampled_pow = sample_valid_subtree("ADD_EXPR", sigma_small=2, rng=random.Random(0))

print(f"One sampled POW subtree proposal: {serialize_prefix_string(sampled_pow)}")
print(f"Original subtree:                 {serialize_prefix_string(original)}")
print(f"Candidate subtree:                {serialize_prefix_string(candidate)}")
print(f"can_sampled_subtree_replace:      {can_sampled_subtree_replace(original, candidate)}")

canonical_original = canonicalize(original)
mutated_raw = replace_subtree_by_node_id(canonical_original, 2, candidate)
mutated = canonicalize(mutated_raw)

print(f"Canonical original:               {serialize_prefix_string(canonical_original)}")
print(f"After replacement + canonicalize: {serialize_prefix_string(mutated)}")


## Full Engine Walkthrough

`mutate_once(...)` is the full mutation engine. In the current implementation it performs this pipeline:

1. canonicalize the input expression,
2. index canonical positions with `index_tree_positions(...)`,
3. group mutable nodes by production family with `collect_candidate_nodes(...)`,
4. choose one family and one node from that family,
5. choose one mutation path for that node: `local_const_edit`, `local_same_arity_replacement`, or `sampled_small_subtree_replacement`,
6. build a concrete replacement subtree,
7. apply it with `replace_subtree_by_node_id(...)`,
8. canonicalize the mutated tree and reject no-op results.

`collect_candidate_nodes(...)` is the bridge between indexing and mutation selection: it runs canonicalization plus indexing and returns the mutable `NodePosition` records grouped by production family, which is exactly the pool `mutate_once(...)` samples from internally.

Two private helpers are worth knowing about when reading the source. `_sample_mutation_kind(...)` decides which of the three mutation paths are legal for the selected node, and `_apply_replacement(...)` performs the replacement plus the final canonicalization and no-op check.

The returned `MutationResult` records both what changed and where it changed in the pre-mutation canonical tree:

- `selected_node_id`: preorder id of the mutated node,
- `selected_family`: production family that was sampled,
- `selected_token_start` / `selected_token_end`: half-open token span in the canonical input serialization,
- `original_subtree`: the canonical subtree that was selected,
- `replacement_subtree`: the raw subtree proposal inserted before final canonicalization,
- `mutated_expr`: the final canonicalized expression returned to the caller.

Because of the final canonicalization step, `mutated_expr` can look more simplified or reordered than the raw `replacement_subtree` might suggest.


In [ ]:
def summarize_candidate_pool(expression: str, sigma_small: int) -> dict[str, list[dict]]:
    canonical_expr = canonicalize(parse_prefix_string(expression))
    index = index_tree_positions(canonical_expr, sigma_small=sigma_small)
    candidates = collect_candidate_nodes(canonical_expr, sigma_small=sigma_small)
    return {
        family: [
            {
                "node_id": position.node_id,
                "span": (position.token_start, position.token_end),
                "subtree_size": position.subtree_size,
                "subtree": serialize_prefix_string(index.node_id_to_node[position.node_id]),
            }
            for position in positions
        ]
        for family, positions in sorted(candidates.items())
    }


def summarize_mutation_result(result) -> dict:
    return {
        "selected_node_id": result.selected_node_id,
        "selected_family": result.selected_family,
        "selected_token_start": result.selected_token_start,
        "selected_token_end": result.selected_token_end,
        "original_subtree": serialize_prefix_string(result.original_subtree),
        "replacement_subtree": serialize_prefix_string(result.replacement_subtree),
        "mutated_expr": serialize_prefix_string(result.mutated_expr),
    }


def run_mutate_once(expression: str, sigma_small: int, seed: int) -> None:
    candidate_pool = summarize_candidate_pool(expression, sigma_small)
    result = mutate_once(parse_prefix_string(expression), sigma_small=sigma_small, rng=random.Random(seed))
    if result is None:
        raise RuntimeError(f"mutate_once returned None for {expression}")

    print("-" * 80)
    print(f"Input expression: {expression}")
    print(f"sigma_small:      {sigma_small}")
    print(f"seed:             {seed}")
    print("Candidate pool by family:")
    pprint(candidate_pool)
    print("MutationResult:")
    pprint(summarize_mutation_result(result))


run_mutate_once("pow x INT+ 2", sigma_small=2, seed=0)
run_mutate_once("add sin x pow x INT+ 2", sigma_small=2, seed=0)


## Mutation Kind Coverage

The earlier sections show the mechanics of local replacement and sampled subtree replacement. The next cell pins one deterministic `mutate_once(...)` example for each current mutation kind emitted by the engine:

- `local_const_edit`
- `local_same_arity_replacement`
- `sampled_small_subtree_replacement`

`MutationResult` does not currently carry a `mutation_kind` field, so the notebook infers the kind from the same legality predicates used by the engine. Each row below asserts that the observed result matches the expected mutation kind.


In [ ]:
from src.tree_diffusion.mutation import (
    LOCAL_CONST_EDIT,
    LOCAL_SAME_ARITY_REPLACEMENT,
    SAMPLED_SMALL_SUBTREE_REPLACEMENT,
)


def infer_mutation_kind(original_subtree, replacement_subtree) -> str:
    if can_locally_replace(original_subtree, replacement_subtree):
        if isinstance(original_subtree, Const) and isinstance(replacement_subtree, Const):
            same_const_leaf_kind = (
                (original_subtree.is_numeric and replacement_subtree.is_numeric)
                or (original_subtree.is_named and replacement_subtree.is_named)
            )
            if same_const_leaf_kind:
                return LOCAL_CONST_EDIT
        return LOCAL_SAME_ARITY_REPLACEMENT

    if can_sampled_subtree_replace(original_subtree, replacement_subtree):
        return SAMPLED_SMALL_SUBTREE_REPLACEMENT

    raise AssertionError("Could not infer a legal mutation kind for the notebook demo.")


MUTATION_KIND_DEMOS = [
    {
        "mutation_kind": LOCAL_CONST_EDIT,
        "expression": "pow x INT+ 5",
        "sigma_small": 0,
        "seed": 1,
    },
    {
        "mutation_kind": LOCAL_SAME_ARITY_REPLACEMENT,
        "expression": "sin x",
        "sigma_small": 1,
        "seed": 2,
    },
    {
        "mutation_kind": SAMPLED_SMALL_SUBTREE_REPLACEMENT,
        "expression": "pow x INT+ 2",
        "sigma_small": 2,
        "seed": 64,
    },
]

mutation_rows = []
for demo in MUTATION_KIND_DEMOS:
    result = mutate_once(
        parse_prefix_string(demo["expression"]),
        sigma_small=demo["sigma_small"],
        rng=random.Random(demo["seed"]),
    )
    if result is None:
        raise RuntimeError(f"mutate_once returned None for {demo}")

    observed_kind = infer_mutation_kind(result.original_subtree, result.replacement_subtree)
    assert observed_kind == demo["mutation_kind"], (demo, observed_kind)

    mutation_rows.append(
        {
            "mutation_kind": observed_kind,
            "expression": demo["expression"],
            "sigma_small": demo["sigma_small"],
            "seed": demo["seed"],
            "selected_node_id": result.selected_node_id,
            "selected_family": result.selected_family,
            "original_subtree": serialize_prefix_string(result.original_subtree),
            "replacement_subtree": serialize_prefix_string(result.replacement_subtree),
            "mutated_expr": serialize_prefix_string(result.mutated_expr),
        }
    )

print("One deterministic mutate_once(...) example for each current mutation kind:")
print_table(
    mutation_rows,
    [
        "mutation_kind",
        "expression",
        "sigma_small",
        "seed",
        "selected_node_id",
        "selected_family",
        "original_subtree",
        "replacement_subtree",
        "mutated_expr",
    ],
)


## Observation State

Phase 3 adds the current observation object that will later become the model input. Given a target integrand `f` and a current antiderivative candidate `I_t`, `build_observation(...)` packages the current canonical antiderivative, its current derivative `g_t = d/dx I_t`, an optional symbolic residual `simplify(g_t - f)`, and optional numeric probe features.

This observation is intentionally inference-time only. It includes the target integrand `f` because symbolic integration is a conditional task, but it does **not** include the reverse edit path or the gold target antiderivative label. Those remain separate supervision targets for training.


In [ ]:
OBS_TARGET_INTEGRAND = "add pow x INT+ 2 cos x"
OBS_GOLD_ANTIDERIVATIVE = "add div pow x INT+ 3 INT+ 3 sin x"
OBS_SIGMA_SMALL = 2
OBS_MUTATION_SEED = 0


def as_prefix(expr) -> str | None:
    return None if expr is None else serialize_prefix_string(expr)


def show_observation(label: str, observation) -> None:
    print("-" * 80)
    print(label)
    print(f"target integrand:       {as_prefix(observation.target_integrand)}")
    print(f"current antiderivative: {as_prefix(observation.current_antiderivative)}")
    print(f"current derivative:     {as_prefix(observation.current_derivative)}")
    print(f"symbolic residual:      {as_prefix(observation.symbolic_residual)}")
    print(f"residual_mode:          {observation.residual_mode}")
    print(f"status:                 {observation.status}")
    print(f"warnings:               {list(observation.warnings)}")
    if observation.numeric_probes is None:
        print("numeric probes:         <disabled>")
        return

    probes = observation.numeric_probes
    print(f"probe points:           {probes.probe_points}")
    print(f"mean_abs_residual:      {probes.mean_abs_residual}")
    print(f"mean_squared_residual:  {probes.mean_squared_residual}")
    print(f"max_abs_residual:       {probes.max_abs_residual}")
    print(f"fraction_finite:        {probes.fraction_finite}")

    probe_rows = [
        {"x": point, "residual": value, "finite": finite}
        for point, value, finite in zip(
            probes.probe_points,
            probes.residual_values,
            probes.finite_mask,
        )
    ]
    print_table(probe_rows, ["x", "residual", "finite"])


target_integrand_tree = canonicalize(parse_prefix_string(OBS_TARGET_INTEGRAND))
gold_antiderivative_tree = canonicalize(parse_prefix_string(OBS_GOLD_ANTIDERIVATIVE))
mutation_demo = mutate_once(
    gold_antiderivative_tree,
    sigma_small=OBS_SIGMA_SMALL,
    rng=random.Random(OBS_MUTATION_SEED),
)
if mutation_demo is None:
    raise RuntimeError("Expected a deterministic forward mutation for the observation demo.")

clean_observation = build_observation(
    target_integrand_tree,
    gold_antiderivative_tree,
    residual_mode="both",
)
mutated_observation = build_observation(
    target_integrand_tree,
    mutation_demo.mutated_expr,
    residual_mode="both",
)

print(f"Target integrand f:      {serialize_prefix_string(target_integrand_tree)}")
print(f"Gold antiderivative I*:  {serialize_prefix_string(gold_antiderivative_tree)}")
print(
    "One current candidate I_t from mutate_once(...): "
    f"{serialize_prefix_string(mutation_demo.mutated_expr)}"
)
print(
    "The observation stores f and the current candidate I_t, "
    "but not the reverse edit label back toward I*."
)

show_observation("Clean observation: I_t already differentiates back to f", clean_observation)
show_observation("Corrupted observation: I_t came from one forward mutation", mutated_observation)


In [ ]:
mode_rows = []
for residual_mode in ("none", "symbolic", "numeric", "both"):
    observation = build_observation(
        target_integrand_tree,
        mutation_demo.mutated_expr,
        residual_mode=residual_mode,
    )
    mode_rows.append(
        {
            "residual_mode": residual_mode,
            "has_derivative": observation.current_derivative is not None,
            "has_symbolic_residual": observation.symbolic_residual is not None,
            "has_numeric_probes": observation.numeric_probes is not None,
            "status": observation.status,
            "warnings": ", ".join(observation.warnings) or "-",
        }
    )

print(f"Default numeric probe points: {DEFAULT_PROBE_POINTS}")
print("Residual modes on the same current candidate:")
print_table(
    mode_rows,
    [
        "residual_mode",
        "has_derivative",
        "has_symbolic_residual",
        "has_numeric_probes",
        "status",
        "warnings",
    ],
)


## Reverse Edit Path

Phase 2 adds the supervised reverse-edit target. This is different from simply undoing the last sampled mutation: given a current corrupted tree and the canonical target tree, `first_edit_toward_target(...)` finds the first useful legal repair step on a clean path back to the target.

The first code cell below shows one deterministic example of each current first-step reason returned by `first_edit_toward_target(...)`. The second code cell keeps an end-to-end corruption-and-repair demo using `compute_edit_path(...)`.

Each reverse edit reports the selected preorder node id in the current canonical tree, the canonical token span for that node, the mutation kind, the original subtree, the replacement subtree, and the resulting tree after replacement plus canonicalization.


In [ ]:
from src.tree_diffusion.edit_path import compute_edit_path, first_edit_toward_target, structural_distance


REVERSE_REASON_DEMOS = [
    {
        "reason": "direct_mismatch_target",
        "current": "pow x INT+ 5",
        "target": "pow x INT+ 3",
        "sigma_small": 0,
        "seed": 0,
    },
    {
        "reason": "direct_ancestor_target",
        "current": "add x x",
        "target": "add x pow x INT+ 2",
        "sigma_small": 2,
        "seed": 0,
    },
    {
        "reason": "local_root_operator",
        "current": "sin x",
        "target": "cos x",
        "sigma_small": 0,
        "seed": 0,
    },
    {
        "reason": "direct_child_target",
        "current": "add x pow x INT+ 3",
        "target": "add sin x pow x INT+ 2",
        "sigma_small": 1,
        "seed": 0,
    },
    {
        "reason": "target_family_intermediate",
        "current": "pow x INT+ 2",
        "target": "pow x add x pow x INT+ 2",
        "sigma_small": 2,
        "seed": 0,
    },
]

reverse_rows = []
for demo in REVERSE_REASON_DEMOS:
    current_tree = canonicalize(parse_prefix_string(demo["current"]))
    target_tree = canonicalize(parse_prefix_string(demo["target"]))
    edit = first_edit_toward_target(
        current_tree,
        target_tree,
        sigma_small=demo["sigma_small"],
        rng=random.Random(demo["seed"]),
    )
    if edit is None:
        raise RuntimeError(f"first_edit_toward_target returned None for {demo}")

    assert edit.reason == demo["reason"], (demo, edit.reason)
    before = structural_distance(current_tree, target_tree)
    after = structural_distance(edit.resulting_tree, target_tree)
    assert after < before, (demo, before, after)

    reverse_rows.append(
        {
            "reason": edit.reason,
            "mutation_kind": edit.mutation_kind,
            "sigma_small": demo["sigma_small"],
            "selected_node_id": edit.selected_node_id,
            "selected_node_span": edit.selected_node_span,
            "current": serialize_prefix_string(current_tree),
            "target": serialize_prefix_string(target_tree),
            "original_subtree": serialize_prefix_string(edit.original_subtree),
            "replacement_subtree": serialize_prefix_string(edit.replacement_subtree),
            "resulting_tree": serialize_prefix_string(edit.resulting_tree),
            "distance": f"{before} -> {after}",
        }
    )

print("One deterministic first_edit_toward_target(...) example for each current reverse-edit reason:")
print_table(
    reverse_rows,
    [
        "reason",
        "mutation_kind",
        "sigma_small",
        "selected_node_id",
        "selected_node_span",
        "current",
        "target",
        "original_subtree",
        "replacement_subtree",
        "resulting_tree",
        "distance",
    ],
)


## Full Reverse Path Demo

The final reverse-edit demo now uses a more structured target expression with nested `mul`, `add`, `sin`, and `pow` nodes. It applies four valid forward mutations with `mutate_once(...)`, then runs `compute_edit_path(...)` to show the sequence of corrective `EditTarget` objects that returns the corrupted tree to the original target.


In [ ]:
from src.tree_diffusion.edit_path import compute_edit_path, structural_distance


REVERSE_DEMO_EXPRESSION = "mul add x pow x INT+ 2 add sin x INT+ 1"
REVERSE_DEMO_SIGMA_SMALL = 2
REVERSE_DEMO_MUTATION_STEPS = 4
REVERSE_DEMO_SEED = 0

target_tree = canonicalize(parse_prefix_string(REVERSE_DEMO_EXPRESSION))
current_tree = target_tree
forward_rng = random.Random(REVERSE_DEMO_SEED)
forward_mutations = []

for step in range(REVERSE_DEMO_MUTATION_STEPS):
    mutation = mutate_once(current_tree, sigma_small=REVERSE_DEMO_SIGMA_SMALL, rng=forward_rng)
    if mutation is None:
        raise RuntimeError(f"mutate_once returned None at forward step {step + 1}")
    forward_mutations.append(mutation)
    current_tree = mutation.mutated_expr

reverse_path = compute_edit_path(
    current_tree,
    target_tree,
    REVERSE_DEMO_SIGMA_SMALL,
    rng=random.Random(REVERSE_DEMO_SEED + 100),
    max_steps=32,
)

print("Target tree:    ", serialize_prefix_string(target_tree))
print("Corrupted tree: ", serialize_prefix_string(current_tree))
print("Initial reverse distance:", structural_distance(current_tree, target_tree))
print()
print("Forward mutations:")
for step, mutation in enumerate(forward_mutations, start=1):
    print(
        f"{step}. node={mutation.selected_node_id} "
        f"{serialize_prefix_string(mutation.original_subtree)} -> "
        f"{serialize_prefix_string(mutation.replacement_subtree)} "
        f"=> {serialize_prefix_string(mutation.mutated_expr)}"
    )

print()
print("Reverse edit path:")
if not reverse_path:
    print("No reverse edit was needed; canonical trees were already equal.")

for step, edit in enumerate(reverse_path, start=1):
    previous_tree = current_tree if step == 1 else reverse_path[step - 2].resulting_tree
    before = structural_distance(previous_tree, target_tree)
    after = structural_distance(edit.resulting_tree, target_tree)
    print(
        f"{step}. reason={edit.reason} kind={edit.mutation_kind} "
        f"node={edit.selected_node_id} span={edit.selected_node_span} "
        f"{serialize_prefix_string(edit.original_subtree)} -> "
        f"{serialize_prefix_string(edit.replacement_subtree)} "
        f"=> {serialize_prefix_string(edit.resulting_tree)} "
        f"(distance {before} -> {after})"
    )

assert reverse_path, "Expected at least one reverse edit in this demo."
assert reverse_path[-1].resulting_tree == target_tree
print()
print("Recovered target:", reverse_path[-1].resulting_tree == target_tree)


## Closing Notes

- All mutation entry points operate on canonicalized trees, so equality, node ids, token spans, and subtree sizes are defined on normalized expressions rather than raw parser output.
- `index_tree_positions(...)` assigns preorder ids and canonical token spans, and `sigma_small` limits mutable nodes by `subtree_size`.
- Local replacement preserves node class, arity, and existing children; only the root label or leaf value or kind changes.
- Sampled subtree replacement preserves production-family compatibility but can change internal descendant structure inside the sampled size budget.
- `mutate_once(...)` currently has three mutation kinds, and the notebook now shows one deterministic example of each.
- `build_observation(...)` packages the inference-time state from `(f, I_t)` into `current_derivative`, an optional symbolic residual, and optional numeric probe features.
- Observation intentionally excludes reverse edit labels and the gold antiderivative; those stay separate from the model input and belong to the supervised training target.
- `first_edit_toward_target(...)` currently has five first-step reasons, and the notebook now shows one deterministic example of each before the full multi-step repair demo.
- `MutationResult` points back to the selected location in the pre-mutation canonical tree, while the returned `mutated_expr` is canonicalized again and may therefore look reordered or simplified.
